# Montra Guest Experience Analytics & Sentiment Intelligence

## Data extraction, quality assessment, and exploratory analysis

**Purpose:** extract the hotel, guest, review, and booking tables; assess data completeness and consistency; explore key patterns; and create a joined dataset for downstream text analytics.

**Workflow**
1. Connect to the project database and load the four source tables.
2. Inspect structure, missing values, duplicate records, identifiers, and date fields.
3. Explore booking, revenue, review, hotel, and guest patterns.
4. Join related tables and save the merged dataset.
5. Generate quality reports, validate relationships and business rules, and document issues.

**Important:** this notebook reports potential issues; it does not prove that every source value is correct. Confirm business rules against the project data dictionary. Keep raw source tables unchanged and apply cleaning to separate copies where appropriate.


## 1. Install the database driver

Install the PostgreSQL driver used by SQLAlchemy to connect to the project database. If it is already installed in your environment, this cell may complete quickly.

In [1]:
# Install the PostgreSQL database driver required by SQLAlchemy.
%pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


## 2. Import libraries

Import only the libraries used in this notebook. `pandas` handles tabular data, `numpy` supports numeric checks, `SQLAlchemy` manages the database connection, and Plotly is used for interactive charts.

In [2]:
# Import libraries used for database access, data cleaning, validation, and charts.
import pandas as pd
import numpy as np
import os

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import plotly.express as px


## 3. Configure the database connection

Load the database URL from the `.env` file rather than hard-coding credentials in the notebook. Keep `.env` private and do not commit it to GitHub. If the URL is not loaded, stop here and check the filename and working directory.

In [3]:
# Load database credentials from the local .env file; never hard-code secrets here.
env_file = Path.cwd() / ".env"
load_dotenv(env_file, override=True)

Database_url = os.getenv("DATABASE_URL")

print("URL loaded successfully:", Database_url is not None)

URL loaded successfully: True


## 4. Test the database connection

A simple `SELECT 1` confirms whether the database can be reached. If this fails, resolve the connection or credential error before trying to load tables.

In [4]:
# Create the SQLAlchemy engine with SSL enabled and a short connection timeout.
# The try/except below prints a readable error if the database cannot be reached.
engine = create_engine(
    Database_url,
    poolclass=None,
    connect_args={
        "sslmode": "require",
        "connect_timeout": 10
    }
)

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("Database response:", result.fetchone())
        print("Successfully connected to db")

except Exception as e:
    print("Connection failed:", repr(e))

Database response: (1,)
Successfully connected to db


## 5. Confirm which tables are available

Query PostgreSQL metadata to list accessible base tables. Compare the result with the four expected project tables before loading data.

In [5]:
# Ask PostgreSQL for the base tables visible to this connection.
query = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'
  AND table_schema NOT IN (
      'pg_catalog',
      'information_schema'
  )
ORDER BY table_schema, table_name;
"""

with engine.connect() as connection:
    tables_df = pd.read_sql(text(query), connection)

display(tables_df)

,table_schema,table_name
0,public,bookings
1,public,clean_bookings
2,public,guests
3,public,hotels
4,public,reviews


## 6. Extract the source tables

Load each source table into a DataFrame. The raw DataFrames (`hotel`, `guest`, `review`, and `booking`) are retained as the baseline for analysis and validation.

In [6]:
# Extract each source table into a pandas DataFrame for analysis.
hotel = pd.read_sql(
    "SELECT * FROM hotels", engine
)

In [7]:
# Load the guests data
guest = pd.read_sql(
    "SELECT * FROM guests", engine
)


In [8]:
# Load the reviews data
review = pd.read_sql(
    "SELECT * FROM reviews", engine
)

In [9]:
# load the bookings data
booking = pd.read_sql(
    "SELECT * from bookings", engine
)
booking.head()

,booking_id,guest_id,hotel_id,room_type,booking_channel,check_in_date,check_out_date,total_amount,stay_duration
0,1,6699,18,Suite,Direct Website,2025-05-25,2025-05-26,296.44,1
1,2,11380,23,Deluxe,Corporate Portal,2025-01-09,2025-01-13,483.56,4
2,3,4812,13,Executive,Travel Agent,2025-03-04,2025-03-06,415.59,2
3,4,982,12,Family Room,Airbnb,2026-03-06,2026-03-09,601.18,3
4,5,615,5,Deluxe,Expedia,2025-02-25,2025-03-02,603.85,5


## 7. Initial data inspection and quality checks

These checks establish the size and structure of each table, identify missing values, and highlight possible duplicate records or suspicious values. Repeated foreign keys in reviews and bookings are normal; primary keys should identify individual records uniquely.

# Data inspection and quality checks

In [10]:
# Keep the raw source DataFrames in one mapping so common checks can be reused.
# These are the original extracts; use separate *_clean copies for any cleaning.
data_tables = {
    "hotels": hotel,
    "guests": guest,
    "reviews": review,
    "bookings": booking,
}

# 1. Dataset dimensions: number of rows and columns in each table.
dimensions = pd.DataFrame([
    {
        "table": table_name,
        "rows": data.shape[0],
        "columns": data.shape[1],
    }
    for table_name, data in data_tables.items()
])

display(dimensions)


,table,rows,columns
0,hotels,25,5
1,guests,12000,5
2,reviews,35000,8
3,bookings,35000,9


In [11]:
# 2. Inspect column names, data types, and a small sample from each table.
# This helps confirm the actual database schema before writing cleaning or join logic.
for table_name, data in data_tables.items():
    print(f"\n{'=' * 70}\n{table_name.upper()} — first 5 rows")
    display(data.head())

    print(f"Column names in {table_name}:")
    print(data.columns.tolist())

    print(f"Data types and non-null counts for {table_name}:")
    data.info()



HOTELS — first 5 rows


,hotel_id,hotel_name,location,star_rating,total_rooms
0,1,The Birmingham Grand,Birmingham,4,254
1,2,The London Heritage,London,5,234
2,3,The Oxford Boutique,Oxford,3,146
3,4,The Glasgow Heritage,Glasgow,3,84
4,5,Glasgow Metropolitan,Glasgow,4,214


Column names in hotels:
['hotel_id', 'hotel_name', 'location', 'star_rating', 'total_rooms']
Data types and non-null counts for hotels:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   hotel_id     25 non-null     int64 
 1   hotel_name   25 non-null     object
 2   location     25 non-null     object
 3   star_rating  25 non-null     int64 
 4   total_rooms  25 non-null     int64 
dtypes: int64(3), object(2)
memory usage: 1.1+ KB

GUESTS — first 5 rows


,guest_id,customer_type,loyalty_status,country,previous_stays
0,1,Leisure,None,Australia,0
1,2,Leisure,None,Germany,1
2,3,Business,Gold,United Kingdom,0
3,4,Family,Silver,United Arab Emirates,8
4,5,Corporate,None,France,0


Column names in guests:
['guest_id', 'customer_type', 'loyalty_status', 'country', 'previous_stays']
Data types and non-null counts for guests:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   guest_id        12000 non-null  int64 
 1   customer_type   12000 non-null  object
 2   loyalty_status  5964 non-null   object
 3   country         12000 non-null  object
 4   previous_stays  12000 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 468.9+ KB

REVIEWS — first 5 rows


,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral


Column names in reviews:
['review_id', 'booking_id', 'guest_id', 'hotel_id', 'review_text', 'rating_score', 'review_date', 'sentiment_label']
Data types and non-null counts for reviews:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   review_id        35000 non-null  int64  
 1   booking_id       35000 non-null  int64  
 2   guest_id         35000 non-null  int64  
 3   hotel_id         35000 non-null  int64  
 4   review_text      35000 non-null  object 
 5   rating_score     35000 non-null  float64
 6   review_date      35000 non-null  object 
 7   sentiment_label  35000 non-null  object 
dtypes: float64(1), int64(4), object(3)
memory usage: 2.1+ MB

BOOKINGS — first 5 rows


,booking_id,guest_id,hotel_id,room_type,booking_channel,check_in_date,check_out_date,total_amount,stay_duration
0,1,6699,18,Suite,Direct Website,2025-05-25,2025-05-26,296.44,1
1,2,11380,23,Deluxe,Corporate Portal,2025-01-09,2025-01-13,483.56,4
2,3,4812,13,Executive,Travel Agent,2025-03-04,2025-03-06,415.59,2
3,4,982,12,Family Room,Airbnb,2026-03-06,2026-03-09,601.18,3
4,5,615,5,Deluxe,Expedia,2025-02-25,2025-03-02,603.85,5


Column names in bookings:
['booking_id', 'guest_id', 'hotel_id', 'room_type', 'booking_channel', 'check_in_date', 'check_out_date', 'total_amount', 'stay_duration']
Data types and non-null counts for bookings:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   booking_id       35000 non-null  int64  
 1   guest_id         35000 non-null  int64  
 2   hotel_id         35000 non-null  int64  
 3   room_type        35000 non-null  object 
 4   booking_channel  35000 non-null  object 
 5   check_in_date    35000 non-null  object 
 6   check_out_date   35000 non-null  object 
 7   total_amount     35000 non-null  float64
 8   stay_duration    35000 non-null  int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 2.4+ MB


In [12]:
# Display all unique hotel locations
print(hotel['location'].unique())

['Birmingham' 'London' 'Oxford' 'Glasgow' 'Leeds' 'Belfast' 'Edinburgh'
 'Southampton' 'Nottingham' 'Manchester' 'Liverpool' 'Leicester'
 'Aberdeen' 'Brighton']


In [13]:
# Convert date fields to pandas datetime values so sorting, resampling, and date checks work.
# errors='coerce' turns invalid date strings into NaT, which can then be reported.
booking["check_in_date"] = pd.to_datetime(booking["check_in_date"], errors="coerce")
booking["check_out_date"] = pd.to_datetime(booking["check_out_date"], errors="coerce")
review["review_date"] = pd.to_datetime(review["review_date"], errors="coerce")

# Invalid date strings become NaT; inspect missing dates in the quality reports before analysis.

In [14]:
# 3. Count missing values per column and calculate their percentage.
# Missing data is reported first; do not automatically drop rows or fill values
# until the business meaning of each field is understood.
for table_name, data in data_tables.items():
    print(f"\nMissing-value report: {table_name}")

    missing_report = pd.DataFrame({
        "missing_count": data.isna().sum(),
        "missing_percent": (data.isna().mean() * 100).round(2),
    })

    # Display only columns that contain at least one missing value.
    missing_report = missing_report[missing_report["missing_count"] > 0]

    if missing_report.empty:
        print("No missing values found.")
    else:
        display(missing_report.sort_values("missing_count", ascending=False))



Missing-value report: hotels
No missing values found.

Missing-value report: guests


,missing_count,missing_percent
loyalty_status,6036,50.3



Missing-value report: reviews
No missing values found.

Missing-value report: bookings
No missing values found.


In [15]:
# Create a separate guest copy so the raw extract remains unchanged.
guest_clean = guest.copy()

In [16]:
# Record the number of missing loyalty statuses before applying the documented treatment.
print("Missing values before cleaning:")
print(guest_clean["loyalty_status"].isna().sum())

Missing values before cleaning:
6036


In [17]:
# Use 'Unknown' to distinguish an unrecorded status from a known loyalty tier.
guest_clean["loyalty_status"] = (
    guest_clean["loyalty_status"].fillna("Unknown")
)

In [18]:
# Step 4: Verify that missing values have been handled
print("\nMissing values after cleaning:")
print(guest_clean["loyalty_status"].isna().sum())


Missing values after cleaning:
0


In [19]:
# Step 5: Examine the distribution of loyalty status
print("\nLoyalty status distribution:")
display(
    guest_clean["loyalty_status"]
    .value_counts(dropna=False)
    .to_frame("guest_count")
)


Loyalty status distribution:


,guest_count
loyalty_status,
Unknown,6036
Silver,2951
Gold,1998
Platinum,1015


In [20]:
# Exact duplicate rows are different from repeated foreign keys; inspect them before removal.
# A duplicate row is not always the same as a duplicate entity ID, so both should be checked.
duplicate_report = pd.DataFrame([
    {
        "table": table_name,
        "exact_duplicate_rows": int(data.duplicated().sum()),
    }
    for table_name, data in data_tables.items()
])

display(duplicate_report)


,table,exact_duplicate_rows
0,hotels,0
1,guests,0
2,reviews,0
3,bookings,0


In [21]:
# Check identifier completeness and repeated values. Interpret uniqueness using each table's key definition.
# this list to the exact column names used in your database.
id_columns = [
    "hotel_id", "guest_id", "review_id", "booking_id",
    "property_id", "customer_id", "reservation_id", "id"
]

for table_name, data in data_tables.items():
    print(f"\nIdentifier checks: {table_name}")
    found_id_columns = [col for col in id_columns if col in data.columns]

    if not found_id_columns:
        print("No common ID column detected. Review the column names above.")
        continue

    for col in found_id_columns:
        print(
            f"{col}: missing={int(data[col].isna().sum())}, "
            f"duplicate values={int(data[col].duplicated().sum())}"
        )



Identifier checks: hotels
hotel_id: missing=0, duplicate values=0

Identifier checks: guests
guest_id: missing=0, duplicate values=0

Identifier checks: reviews
hotel_id: missing=0, duplicate values=34975
guest_id: missing=0, duplicate values=23623
review_id: missing=0, duplicate values=0
booking_id: missing=0, duplicate values=0

Identifier checks: bookings
hotel_id: missing=0, duplicate values=34975
guest_id: missing=0, duplicate values=23623
booking_id: missing=0, duplicate values=0


In [22]:
# Summarise numeric columns to help spot impossible values and outliers; do not auto-correct them.
# This is a diagnostic summary, not an automatic correction.
for table_name, data in data_tables.items():
    numeric_data = data.select_dtypes(include=[np.number])
    print(f"\nNumeric summary: {table_name}")

    if numeric_data.empty:
        print("No numeric columns found.")
    else:
        display(numeric_data.describe().T)



Numeric summary: hotels


,count,mean,std,min,25%,50%,75%,max
hotel_id,25.0,13.00,7.359801,1.0,7.0,13.0,19.0,25.0
star_rating,25.0,3.72,0.737111,3.0,3.0,4.0,4.0,5.0
total_rooms,25.0,173.92,63.988228,71.0,116.0,197.0,224.0,265.0



Numeric summary: guests


,count,mean,std,min,25%,50%,75%,max
guest_id,12000.0,6000.500,3464.245950,1.0,3000.75,6000.5,9000.25,12000.0
previous_stays,12000.0,2.107,2.249809,0.0,0.00,2.0,4.00,12.0



Numeric summary: reviews


,count,mean,std,min,25%,50%,75%,max
review_id,35000.0,17500.500000,10103.774047,1.0,8750.75,17500.5,26250.25,35000.0
booking_id,35000.0,17500.500000,10103.774047,1.0,8750.75,17500.5,26250.25,35000.0
guest_id,35000.0,5984.449571,3470.140340,1.0,2957.00,6002.0,8986.00,11999.0
hotel_id,35000.0,12.973943,7.190291,1.0,7.00,13.0,19.00,25.0
rating_score,35000.0,3.027429,0.931849,1.0,2.40,3.0,3.70,5.0



Numeric summary: bookings


,count,mean,std,min,25%,50%,75%,max
booking_id,35000.0,17500.500000,10103.774047,1.00,8750.7500,17500.50,26250.250,35000.00
guest_id,35000.0,5984.449571,3470.140340,1.00,2957.0000,6002.00,8986.000,11999.00
hotel_id,35000.0,12.973943,7.190291,1.00,7.0000,13.00,19.000,25.00
total_amount,35000.0,466.057135,315.050703,76.53,241.3575,379.59,599.495,2447.18
stay_duration,35000.0,2.954514,1.503233,1.00,2.0000,3.00,4.000,7.00


## 8. Exploratory data analysis (EDA)

EDA summarises the data before modelling. The charts help identify booking patterns, revenue differences, review behaviour, hotel distribution, and guest characteristics. Treat patterns as descriptive evidence, not proof of cause and effect.

Exploratory Data Analysis


In [23]:
# Import the required libraries
# Import Plotly Express for quick interactive visualisations
import plotly.express as px

# Import Plotly Graph Objects for custom charts
import plotly.graph_objects as go

# Import make_subplots to combine multiple charts into one figure
from plotly.subplots import make_subplots

In [24]:
# Analyse bookings by room type
# Total booking count by room type
# Count how many bookings belong to each room type.
room_type = booking['room_type'].value_counts().reset_index()
# Display the room_type
room_type

,room_type,count
0,Standard,10596
1,Deluxe,8619
2,Executive,5293
3,Family Room,4137
4,Suite,3544
5,Twin,2811


In [25]:
# Revenue generated by room type
# Group bookings by room type and sum the total amount for each type.
room_revenue = (
    booking.groupby('room_type')['total_amount']
    .sum()
    .reset_index()
)
# Display the revenue summary.
room_revenue


,room_type,total_amount
0,Deluxe,3938732.66
1,Executive,3211430.94
2,Family Room,2158312.62
3,Standard,3112788.77
4,Suite,3029593.14
5,Twin,861141.58


In [26]:
# Analyse bookings by booking channel
# This analysis helps identify which room types are booked most frequently and which generate the most revenue.
# Booking Channel Distribution
# Count the number of bookings made through each booking channel.
channels = booking['booking_channel'].value_counts().reset_index()
# Display the booking channel counts.
channels



,booking_channel,count
0,Direct Website,8688
1,Booking.com,7808
2,Expedia,5273
3,Travel Agent,3529
4,Airbnb,3416
5,Corporate Portal,3411
6,Walk-in,2875


In [27]:
# Revenue by booking channel
# Calculate total revenue generated by each booking channel.
channel_revenue = (
    booking.groupby('booking_channel')['total_amount']
    .sum()
    .reset_index()
)
# Display the revenue summary by booking channel.
channel_revenue

,booking_channel,total_amount
0,Airbnb,1604609.36
1,Booking.com,3654237.93
2,Corporate Portal,1611396.00
3,Direct Website,4051385.20
4,Expedia,2452361.55
5,Travel Agent,1618571.44
6,Walk-in,1319438.23


In [28]:
# Create a 2 × 2 subplot dashboard
# The following code combines the four analyses into one interactive figure

# Create a subplot figure with 2 rows and 2 columns.
# Each subplot will display a different booking analysis.

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Total Room Type Distribution",
        "Total Revenue Generated by Room Type",
        "Booking Channel Distribution",
        "Total Revenue Generated by Booking Channel"
    )
)


# Chart 1: Booking count by room type
# A bar chart showing the number of bookings for each room type.
fig.add_trace(
    go.Bar(
        x=room_type['room_type'],
        y=room_type['count'],
        name='Count',
        marker_color='teal'
    ),
    row=1,
    col=1
)


# Chart 2: Revenue by room type
# Compare the revenue generated by different room types.
fig.add_trace(
    go.Bar(
        x=room_revenue['room_type'],
        y=room_revenue['total_amount'],
        name='Revenue',
        marker_color='indianred'
    ),
    row=1,
    col=2
)


# Chart 3: Booking channel distribution
# Show the number of bookings made through each channel.
fig.add_trace(
    go.Bar(
        x=channels['booking_channel'],
        y=channels['count'],
        name='Channel Count',
        marker_color='blue'
    ),
    row=2,
    col=1
)


# Chart 4: Revenue by booking channel
# Compare the total revenue associated with each channel.
fig.add_trace(
    go.Bar(
        x=channel_revenue['booking_channel'],
        y=channel_revenue['total_amount'],
        name='Channel Revenue',
        marker_color='goldenrod'
    ),
    row=2,
    col=2
)


# Update the overall layout of the dashboard.
fig.update_layout(
    height=800,
    title_text='Booking Analysis Overview',
    showlegend=False
)

# Display the interactive dashboard.
fig.show()

In [29]:
# Format revenue values as currency when hovering over the bars.
fig.update_traces(
    hovertemplate='%{x}<br>Revenue: £%{y:,.2f}<extra></extra>',
    selector=dict(type='bar'),
    row=1,
    col=2
)

fig.update_traces(
    hovertemplate='%{x}<br>Revenue: £%{y:,.2f}<extra></extra>',
    selector=dict(type='bar'),
    row=2,
    col=2
)

# Render the updated dashboard.
fig.show()

### Review analysis

Review-level EDA examines rating and sentiment distributions, review length, and changes in review volume over time. These summaries help assess the available text and labels before NLP preprocessing.

# review analyses: 

In [30]:

# Create a column showing the number of characters in each review
review['review_length'] = review['review_text'].fillna('').str.len()

# Review rating distribution
rating_distribution = (
    review['rating_score'].value_counts().sort_index().reset_index()
)
rating_distribution.columns = ['rating_score', 'count']

# Review sentiment distribution
sentiment_distribution = (
    review['sentiment_label'].value_counts().reset_index()
)
sentiment_distribution.columns = ['sentiment_label', 'count']

# Monthly review volume
monthly_reviews = (
    review.dropna(subset=['review_date'])
    .set_index('review_date')
    .resample('MS')
    .size()
    .reset_index(name='review_count')
)

# Review length distribution
review_lengths = review['review_length'].dropna()

### Review charts

Display the review charts together so rating scores, sentiment labels, review volume, and review length can be compared in one view.

Display all review EDA charts together

In [54]:
# Create a 2-row, 2-column subplot dashboard
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Review Rating Distribution",
        "Review Sentiment Distribution",
        "Monthly Review Volume",
        "Review Length Distribution"
    )
)

# Chart 1: Distribution of review ratings
fig.add_trace(
    go.Histogram(
        x=review['rating_score'].dropna(),
        nbinsx=20,
        name='Ratings',
        marker_color='teal'
    ),
    row=1, col=1
)

# Chart 2: Distribution of sentiment labels
fig.add_trace(
    go.Bar(
        x=sentiment_distribution['sentiment_label'],
        y=sentiment_distribution['count'],
        name='Sentiment',
        marker_color='royalblue'
    ),
    row=1, col=2
)

# Chart 3: Number of reviews submitted each month
fig.add_trace(
    go.Scatter(
        x=monthly_reviews['review_date'],
        y=monthly_reviews['review_count'],
        mode='lines+markers',
        name='Monthly Reviews',
        line=dict(color='indianred')
    ),
    row=2, col=1
)

# Chart 4: Distribution of review text lengths
fig.add_trace(
    go.Histogram(
        x=review_lengths,
        nbinsx=30,
        name='Review Length',
        marker_color='goldenrod'
    ),
    row=2, col=2
)

# Add axis labels
fig.update_xaxes(title_text='Rating Score', row=1, col=1)
fig.update_yaxes(title_text='Number of Reviews', row=1, col=1)

fig.update_xaxes(title_text='Sentiment', row=1, col=2)
fig.update_yaxes(title_text='Number of Reviews', row=1, col=2)

fig.update_xaxes(title_text='Review Date', row=2, col=1)
fig.update_yaxes(title_text='Number of Reviews', row=2, col=1)

fig.update_xaxes(title_text='Characters per Review', row=2, col=2)
fig.update_yaxes(title_text='Number of Reviews', row=2, col=2)

# Configure the dashboard
fig.update_layout(
    title_text='Montra Hospitality Intel — Guest Review EDA',
    height=600,
    width=900,
    showlegend=False,
    template='plotly_white'
)

# Display all four charts together
fig.show()

In [32]:
# Summarise rating scores and review lengths
display(
    review[['rating_score', 'review_length']].describe()
)

# Show sentiment counts and percentages
sentiment_summary = (
    review['sentiment_label']
    .value_counts(dropna=False)
    .to_frame('count')
)

sentiment_summary['percentage'] = (
    sentiment_summary['count'] / len(review) * 100
).round(2)

display(sentiment_summary)

# Check missing values in each review column
display(
    review.isnull().sum().to_frame('missing_count')
)

,rating_score,review_length
count,35000.000000,35000.000000
mean,3.027429,171.913743
std,0.931849,39.801050
min,1.000000,67.000000
25%,2.400000,143.000000
50%,3.000000,172.000000
75%,3.700000,203.000000
max,5.000000,280.000000


,count,percentage
sentiment_label,,
neutral,13355,38.16
positive,11782,33.66
negative,9863,28.18


,missing_count
review_id,0
booking_id,0
guest_id,0
hotel_id,0
review_text,0
rating_score,0
review_date,0
sentiment_label,0
review_length,0


### Hotel analysis

Inspect hotel attributes and visualise hotel distribution by location and star rating. Check the missing-value report before interpreting the charts.

In [33]:
# Create a copy for EDA
hotel_eda = hotel.copy()

In [34]:
# Inspect hotel data types and missing values
hotel_eda.info()
display(hotel_eda.isnull().sum().to_frame('missing_count'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   hotel_id     25 non-null     int64 
 1   hotel_name   25 non-null     object
 2   location     25 non-null     object
 3   star_rating  25 non-null     int64 
 4   total_rooms  25 non-null     int64 
dtypes: int64(3), object(2)
memory usage: 1.1+ KB


,missing_count
hotel_id,0
hotel_name,0
location,0
star_rating,0
total_rooms,0


In [35]:
# 1. Number of hotels by location
hotels_by_location = (
    hotel_eda['location']
    .value_counts()
    .reset_index()
)
hotels_by_location.columns = ['location', 'hotel_count']

# 2. Distribution of star ratings
star_distribution = (
    hotel_eda['star_rating']
    .value_counts()
    .sort_index()
    .reset_index()
)
star_distribution.columns = ['star_rating', 'hotel_count']

# 3. # Calculate total room capacity by location
rooms_by_location = (
    hotel.groupby('location')['total_rooms']
    .sum()
    .reset_index()
    .sort_values('total_rooms', ascending=True)
)

# 4. Room capacity by star rating
rooms_by_rating = (
    hotel_eda.groupby('star_rating')['total_rooms']
    .sum()
    .reset_index()
    .sort_values('star_rating')
)


In [59]:
#  Display the number of hotels available in each location
print(
    "1. Number of Hotels by Location\n",
    hotels_by_location.to_string(index=False),
    "\n\n2. Distribution of Star Ratings\n",
    star_distribution.to_string(index=False),
    "\n\n3. Total Room Capacity by Location\n",
    rooms_by_location.to_string(index=False),
    "\n\n4. Room Capacity by Star Rating\n",
    rooms_by_rating.to_string(index=False)
)

1. Number of Hotels by Location
    location  hotel_count
    Glasgow            4
     London            3
    Belfast            3
 Birmingham            2
  Edinburgh            2
 Nottingham            2
  Liverpool            2
     Oxford            1
      Leeds            1
Southampton            1
 Manchester            1
  Leicester            1
   Aberdeen            1
   Brighton            1 

2. Distribution of Star Ratings
  star_rating  hotel_count
           3           11
           4           10
           5            4 

3. Total Room Capacity by Location
    location  total_rooms
Southampton           80
 Manchester          101
  Leicester          116
   Aberdeen          118
     Oxford          146
      Leeds          177
   Brighton          209
  Edinburgh          286
 Birmingham          334
 Nottingham          349
  Liverpool          414
    Belfast          636
    Glasgow          652
     London          730 

4. Room Capacity by Star Rating
  star

In [60]:
# Create a 2 x 2 dashboard
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        'Hotel Distribution by Location',
        'Room Capacity Distribution',
        'Total Room Capacity by Location',
        'Hotel Star Rating vs Room Capacity'
    ),
    specs=[
        [{'type': 'domain'}, {'type': 'box'}],
        [{'type': 'bar'}, {'type': 'scatter'}]
    ]
)

# Chart 1: Donut chart — Hotels by location
fig.add_trace(
    go.Pie(
        labels=hotels_by_location['location'],
        values=hotels_by_location['hotel_count'],
        hole=0.5,
        name='Hotels',
        textinfo='label+percent'
    ),
    row=1, col=1
)

# Chart 2: Box plot — Distribution of room capacity
fig.add_trace(
    go.Box(
        y=hotel['total_rooms'],
        name='Room Capacity',
        marker_color='gold',
        boxmean=True
    ),
    row=1, col=2
)

# Chart 3: Horizontal bar chart — Total rooms by location
fig.add_trace(
    go.Bar(
        x=rooms_by_location['total_rooms'],
        y=rooms_by_location['location'],
        orientation='h',
        marker_color='seagreen',
        text=rooms_by_location['total_rooms'],
        textposition='auto',
        name='Total Rooms'
    ),
    row=2, col=1
)

# Chart 4: Scatter plot — Star rating vs room capacity
fig.add_trace(
    go.Scatter(
        x=hotel['star_rating'],
        y=hotel['total_rooms'],
        mode='markers',
        marker=dict(
            color=hotel['star_rating'],
            colorscale='Viridis',
            size=10,
            showscale=True,
            colorbar=dict(title='Star Rating')
        ),
        text=hotel['hotel_name'],
        hovertemplate=(
            'Hotel: %{text}<br>'
            'Star Rating: %{x}<br>'
            'Total Rooms: %{y}<extra></extra>'
        ),
        name='Hotels'
    ),
    row=2, col=2
)

# Update dashboard layout
fig.update_layout(
    height=600,
    width=1000,
    title_text='Montra Hospitality Intel — Hotel EDA',
    showlegend=False,
    template='plotly_white'
)

# Label axes
fig.update_yaxes(title_text='Number of Rooms', row=1, col=2)
fig.update_xaxes(title_text='Total Rooms', row=2, col=1)
fig.update_yaxes(title_text='Location', row=2, col=1)
fig.update_xaxes(title_text='Star Rating', row=2, col=2)
fig.update_yaxes(title_text='Total Rooms', row=2, col=2)

# Display dashboard
fig.show()

### Guest analysis

Explore customer types, loyalty status, country distribution, and previous stays. Missing loyalty status is represented as `Unknown` in a separate cleaned copy; this preserves the distinction between a missing value and a known loyalty tier.

# Guest EDA,

In [37]:
# Check missing values
display(guest.isnull().sum().to_frame('missing_count'))

,missing_count
guest_id,0
customer_type,0
loyalty_status,6036
country,0
previous_stays,0


In [38]:
# Check duplicate guest IDs
print("Duplicate guest IDs:", guest['guest_id'].duplicated().sum())

Duplicate guest IDs: 0


In [39]:
# Summary statistics for previous stays
display(guest[['previous_stays']].describe())

,previous_stays
count,12000.000000
mean,2.107000
std,2.249809
min,0.000000
25%,0.000000
50%,2.000000
75%,4.000000
max,12.000000


In [40]:
# Check unique values
print("Unique customer types:", guest['customer_type'].nunique())
print("Unique loyalty statuses:", guest['loyalty_status'].nunique())
print("Unique countries:", guest['country'].nunique())

Unique customer types: 5
Unique loyalty statuses: 3
Unique countries: 15


In [41]:
# Prepare data for visualisation
# Guest distribution by customer type
customer_type_counts = guest_clean['customer_type'].value_counts().reset_index()
customer_type_counts.columns = ['customer_type', 'count']

# Guest distribution by loyalty status
loyalty_counts = guest['loyalty_status'].fillna('Unknown').value_counts().reset_index()
loyalty_counts.columns = ['loyalty_status', 'count']

# Guest distribution by country (top 10)
country_counts = (
    guest['country'].value_counts()
    .head(10)
    .reset_index()
)
country_counts.columns = ['country', 'count']

# Previous stays distribution
previous_stays = guest['previous_stays'].dropna()

In [63]:
# Display the total room capacity by location
print("\n" + "=" * 60)
print("3. TOTAL ROOM CAPACITY BY LOCATION")
print("=" * 60)
print(rooms_by_location.to_string(index=False))





3. TOTAL ROOM CAPACITY BY LOCATION
   location  total_rooms
Southampton           80
 Manchester          101
  Leicester          116
   Aberdeen          118
     Oxford          146
      Leeds          177
   Brighton          209
  Edinburgh          286
 Birmingham          334
 Nottingham          349
  Liverpool          414
    Belfast          636
    Glasgow          652
     London          730


In [64]:
# Display the total room capacity by star rating
print("\n" + "=" * 60)
print("4. ROOM CAPACITY BY STAR RATING")
print("=" * 60)
print(rooms_by_rating.to_string(index=False))


4. ROOM CAPACITY BY STAR RATING
 star_rating  total_rooms
           3         1678
           4         1749
           5          921


In [65]:
# Create the 2 × 2 EDA dashboard
# Create a 2 x 2 dashboard
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        'Guest Distribution by Customer Type',
        'Guest Loyalty Status',
        'Distribution of Previous Stays',
        'Top 10 Guest Countries'
    ),
    specs=[
        [{'type': 'domain'}, {'type': 'domain'}],
        [{'type': 'histogram'}, {'type': 'bar'}]
    ]
)

# Chart 1: Donut chart — Customer type
fig.add_trace(
    go.Pie(
        labels=customer_type_counts['customer_type'],
        values=customer_type_counts['count'],
        hole=0.45,
        textinfo='label+percent',
        name='Customer Type'
    ),
    row=1, col=1
)

# Chart 2: Donut chart — Loyalty status
fig.add_trace(
    go.Pie(
        labels=loyalty_counts['loyalty_status'],
        values=loyalty_counts['count'],
        hole=0.45,
        textinfo='label+percent',
        marker_colors=[
            {
                'Gold': 'gold',
                'Silver': 'silver',
                'Bronze': '#CD7F32',
                'Unknown': 'lightgrey'
            }.get(str(status), 'royalblue')
            for status in loyalty_counts['loyalty_status']
        ],
        name='Loyalty Status'
    ),
    row=1, col=2
)

# Chart 3: Histogram — Previous stays
fig.add_trace(
    go.Histogram(
        x=previous_stays,
        nbinsx=20,
        marker_color='seagreen',
        name='Previous Stays'
    ),
    row=2, col=1
)

# Chart 4: Horizontal bar chart — Top 10 countries
fig.add_trace(
    go.Bar(
        x=country_counts['count'],
        y=country_counts['country'],
        orientation='h',
        marker_color='indianred',
        text=country_counts['count'],
        textposition='auto',
        name='Guest Countries'
    ),
    row=2, col=2
)

# Update dashboard layout
fig.update_layout(
    height=550,
    width=1000,
    title_text='Montra Hospitality Intel — Guest EDA',
    showlegend=False,
    template='plotly_white'
)

# Label axes
fig.update_xaxes(title_text='Number of Previous Stays', row=2, col=1)
fig.update_yaxes(title_text='Number of Guests', row=2, col=1)

fig.update_xaxes(title_text='Number of Guests', row=2, col=2)
fig.update_yaxes(title_text='Country', row=2, col=2, autorange='reversed')

# Display dashboard
fig.show()

In [43]:
# Guest summary table
# Loyalty status summary
loyalty_summary = guest['loyalty_status'].fillna('Unknown').value_counts().to_frame('count')
loyalty_summary['percentage'] = (
    loyalty_summary['count'] / len(guest) * 100
).round(2)

display(loyalty_summary)

# Customer type summary
customer_summary = guest['customer_type'].value_counts().to_frame('count')
customer_summary['percentage'] = (
    customer_summary['count'] / len(guest) * 100
).round(2)

display(customer_summary)

# Previous stays by customer type
display(
    guest.groupby('customer_type')['previous_stays']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .round(2)
)

,count,percentage
loyalty_status,,
Unknown,6036,50.30
Silver,2951,24.59
Gold,1998,16.65
Platinum,1015,8.46


,count,percentage
customer_type,,
Leisure,4787,39.89
Business,2934,24.45
Corporate,1876,15.63
Group,1205,10.04
Family,1198,9.98


,count,mean,median,min,max
customer_type,,,,,
Business,2934,2.03,1.0,0,11
Corporate,1876,2.09,1.0,0,12
Family,1198,2.14,2.0,0,9
Group,1205,2.16,2.0,0,10
Leisure,4787,2.14,2.0,0,12


## 9. Merge the source tables

Join reviews to their associated booking, guest, and hotel records using the relevant identifiers. An inner join keeps only records with matching keys in all joined tables, so compare the row count before and after joining to understand how many records may have been excluded.

# merge the tables

In [44]:
# Start with reviews and attach booking details using the shared booking, guest, and hotel identifiers.
# Inner joins can drop unmatched rows, so the final row count should be reviewed.
merge = review.merge(
    booking,
    on=['booking_id', 'guest_id', 'hotel_id'],
    how='inner',
    suffixes=('_review', '_booking')
)

# Merge with guest information
merge = merge.merge(
    guest,
    on='guest_id',
    how='inner'
)

# Merge with hotel information
merge = merge.merge(
    hotel,
    on='hotel_id',
    how='inner'
)

# Check the merged dataset
print("Merged dataset shape:", merge.shape)
display(merge.head())

Merged dataset shape: (35000, 23)


,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,review_length,room_type,...,total_amount,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,80,Suite,...,296.44,1,Leisure,None,France,4,The Aberdeen Grand,Aberdeen,4,118
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,160,Deluxe,...,483.56,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,102,Executive,...,415.59,2,Business,None,Ireland,4,The London Boutique,London,4,231
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,129,Family Room,...,601.18,3,Leisure,None,Nigeria,0,Manchester Metropolitan,Manchester,3,101
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,196,Deluxe,...,603.85,5,Group,None,United States,9,Glasgow Metropolitan,Glasgow,4,214


In [45]:
# Summarise missing values in the merged data to see whether joins or source fields need attention.
merge.isna().sum()

review_id              0
booking_id             0
guest_id               0
hotel_id               0
review_text            0
rating_score           0
review_date            0
sentiment_label        0
review_length          0
room_type              0
booking_channel        0
check_in_date          0
check_out_date         0
total_amount           0
stay_duration          0
customer_type          0
loyalty_status     17615
country                0
previous_stays         0
hotel_name             0
location               0
star_rating            0
total_rooms            0
dtype: int64

In [46]:
# Save the merged extract for the next notebook. The output directory is created if needed.
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

merged_path = processed_dir / "merged_raw.csv"
merge.to_csv(merged_path, index=False)

print("Saved merged dataset:", merged_path.resolve())
print("Merged dataset shape:", merge.shape)


Saved merged dataset: C:\Users\telvi\Downloads\AMDARI\montra_guest_exp_sentiment\data\processed\merged_raw.csv
Merged dataset shape: (35000, 23)


## 10. Data quality and reliability validation

The following checks produce auditable reports for missing values, exact duplicate rows, primary-key integrity, orphan foreign keys, and selected domain rules. They intentionally report problems rather than automatically deleting or altering source records.


## Data Quality and Reliability Validation

In [66]:
quality_dir = Path("../data/processed/quality_reports")
quality_dir.mkdir(parents=True, exist_ok=True)

data_tables = {
    "hotels": hotel,
    "guests": guest,
    "reviews": review,
    "bookings": booking,
}


In [67]:
# 1. Profile missing values, exact duplicate rows, and primary-key quality.
missing_rows = []
duplicate_rows = []
primary_key_rows = []

primary_keys = {
    "hotels": "hotel_id",
    "guests": "guest_id",
    "reviews": "review_id",
    "bookings": "booking_id",
}

for table_name, df in data_tables.items():
    for column in df.columns:
        missing_rows.append({
            "table": table_name,
            "column": column,
            "missing_count": int(df[column].isna().sum()),
            "missing_percent": round(float(df[column].isna().mean() * 100), 2),
        })

    duplicate_rows.append({
        "table": table_name,
        "exact_duplicate_rows": int(df.duplicated().sum()),
    })

    key = primary_keys[table_name]
    primary_key_rows.append({
        "table": table_name,
        "primary_key": key,
        "missing_key_count": int(df[key].isna().sum()) if key in df else None,
        "duplicate_key_count": int(df[key].duplicated().sum()) if key in df else None,
        "key_column_exists": key in df.columns,
    })

missing_report = pd.DataFrame(missing_rows)
duplicate_report = pd.DataFrame(duplicate_rows)
primary_key_report = pd.DataFrame(primary_key_rows)

missing_report.to_csv(quality_dir / "missing_values_report.csv", index=False)
duplicate_report.to_csv(quality_dir / "duplicate_rows_report.csv", index=False)
primary_key_report.to_csv(quality_dir / "primary_key_report.csv", index=False)

display(missing_report.query("missing_count > 0").sort_values(
    ["table", "missing_count"], ascending=[True, False]
))
display(duplicate_report)
display(primary_key_report)


,table,column,missing_count,missing_percent
7,guests,loyalty_status,6036,50.3


,table,exact_duplicate_rows
0,hotels,0
1,guests,0
2,reviews,0
3,bookings,0


,table,primary_key,missing_key_count,duplicate_key_count,key_column_exists
0,hotels,hotel_id,0,0,True
1,guests,guest_id,0,0,True
2,reviews,review_id,0,0,True
3,bookings,booking_id,0,0,True


In [68]:
# 2. Validate foreign-key relationships across the source tables.
# Repeated foreign keys are expected; only values with no matching parent are reported.
relationship_specs = [
    ("reviews", "hotel_id", "hotels", "hotel_id"),
    ("reviews", "guest_id", "guests", "guest_id"),
    ("reviews", "booking_id", "bookings", "booking_id"),
    ("bookings", "hotel_id", "hotels", "hotel_id"),
    ("bookings", "guest_id", "guests", "guest_id"),
]

relationship_rows = []
for child_table, child_col, parent_table, parent_col in relationship_specs:
    child = data_tables[child_table]
    parent = data_tables[parent_table]

    if child_col not in child.columns or parent_col not in parent.columns:
        relationship_rows.append({
            "child_table": child_table, "child_column": child_col,
            "parent_table": parent_table, "parent_column": parent_col,
            "orphan_count": None, "status": "NOT CHECKED - column missing",
        })
        continue

    parent_values = set(parent[parent_col].dropna())
    non_null_child = child[child_col].dropna()
    orphan_mask = ~non_null_child.isin(parent_values)
    orphan_count = int(orphan_mask.sum())

    relationship_rows.append({
        "child_table": child_table, "child_column": child_col,
        "parent_table": parent_table, "parent_column": parent_col,
        "orphan_count": orphan_count,
        "status": "PASS" if orphan_count == 0 else "FAIL",
    })

relationship_report = pd.DataFrame(relationship_rows)
relationship_report.to_csv(quality_dir / "foreign_key_report.csv", index=False)
display(relationship_report)


,child_table,child_column,parent_table,parent_column,orphan_count,status
0,reviews,hotel_id,hotels,hotel_id,0,PASS
1,reviews,guest_id,guests,guest_id,0,PASS
2,reviews,booking_id,bookings,booking_id,0,PASS
3,bookings,hotel_id,hotels,hotel_id,0,PASS
4,bookings,guest_id,guests,guest_id,0,PASS


In [69]:
# 3. Check domain rules and date consistency.
# Confirm the 1-5 rating scale with the source data dictionary.
domain_rows = []

def add_range_check(table_name, column, minimum=None, maximum=None):
    df = data_tables[table_name]
    if column not in df.columns:
        domain_rows.append({
            "table": table_name, "column": column, "invalid_count": None,
            "rule": "column exists", "status": "NOT CHECKED - column missing",
        })
        return

    values = pd.to_numeric(df[column], errors="coerce")
    invalid = values.isna() & df[column].notna()
    if minimum is not None:
        invalid |= values < minimum
    if maximum is not None:
        invalid |= values > maximum

    domain_rows.append({
        "table": table_name,
        "column": column,
        "invalid_count": int(invalid.sum()),
        "rule": f"{minimum if minimum is not None else '-inf'} <= value <= {maximum if maximum is not None else 'inf'}",
        "status": "PASS" if not invalid.any() else "FAIL",
    })

add_range_check("hotels", "star_rating", 1, 5)
add_range_check("reviews", "rating_score", 1, 5)
add_range_check("guests", "previous_stays", 0, None)
add_range_check("hotels", "total_rooms", 0, None)
add_range_check("bookings", "total_amount", 0, None)

# Parse dates safely and report invalid or inconsistent booking dates.
booking_dates = booking.copy()
for col in ["check_in_date", "check_out_date"]:
    if col in booking_dates.columns:
        booking_dates[col] = pd.to_datetime(booking_dates[col], errors="coerce")

if {"check_in_date", "check_out_date"}.issubset(booking_dates.columns):
    invalid_dates = (
        booking_dates["check_in_date"].isna()
        | booking_dates["check_out_date"].isna()
    )
    reversed_dates = (
        booking_dates["check_in_date"].notna()
        & booking_dates["check_out_date"].notna()
        & (booking_dates["check_out_date"] < booking_dates["check_in_date"])
    )
    domain_rows.extend([
        {
            "table": "bookings", "column": "check_in_date/check_out_date",
            "invalid_count": int(invalid_dates.sum()),
            "rule": "both dates parse successfully", 
            "status": "PASS" if not invalid_dates.any() else "FAIL",
        },
        {
            "table": "bookings", "column": "check_out_date",
            "invalid_count": int(reversed_dates.sum()),
            "rule": "check-out is on or after check-in",
            "status": "PASS" if not reversed_dates.any() else "FAIL",
        },
    ])

domain_report = pd.DataFrame(domain_rows)
domain_report.to_csv(quality_dir / "domain_rules_report.csv", index=False)
display(domain_report)


,table,column,invalid_count,rule,status
0,hotels,star_rating,0,1 <= value <= 5,PASS
1,reviews,rating_score,0,1 <= value <= 5,PASS
2,guests,previous_stays,0,0 <= value <= inf,PASS
3,hotels,total_rooms,0,0 <= value <= inf,PASS
4,bookings,total_amount,0,0 <= value <= inf,PASS
5,bookings,check_in_date/check_out_date,0,both dates parse successfully,PASS
6,bookings,check_out_date,0,check-out is on or after check-in,PASS


### Great Expectations validation

Great Expectations adds explicit, automated expectations for essential rules such as non-empty tables, non-null primary keys, and unique primary keys. The pandas reports above provide broader diagnostics; the Great Expectations checks are a complementary validation layer.

If the API differs in your installed Great Expectations version, note the version and error message before changing the checks.

## Great Expectations automated validation

This section uses Great Expectations' DataFrame API to validate essential table and key rules. 

In [51]:
%pip install -q great_expectations

Note: you may need to restart the kernel to use updated packages.


In [70]:
# Run Great Expectations checks against each source DataFrame.
# The report captures each expectation result so failures can be reviewed.
import great_expectations as gx

gx_rows = []
context = gx.get_context(mode="ephemeral")

for table_name, df in data_tables.items():
    datasource_name = f"{table_name}_pandas_source"
    asset_name = f"{table_name}_asset"

    datasource = context.data_sources.add_pandas(name=datasource_name)
    asset = datasource.add_dataframe_asset(name=asset_name)
    batch_definition = asset.add_batch_definition_whole_dataframe(
        name=f"{table_name}_whole_dataframe"
    )
    batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

    expectations = [
        (
            "table_has_rows",
            gx.expectations.ExpectTableRowCountToBeBetween(min_value=1),
        ),
    ]

    # All tables should have non-null primary keys, and each primary key should be unique.
    key = primary_keys[table_name]
    if key in df.columns:
        expectations.extend([
            (
                f"{key}_not_null",
                gx.expectations.ExpectColumnValuesToNotBeNull(column=key),
            ),
            (
                f"{key}_unique",
                gx.expectations.ExpectColumnValuesToBeUnique(column=key),
            ),
        ])

    # Review text is expected to exist as a column; null text is recorded as a quality issue.
    if table_name == "reviews" and "review_text" in df.columns:
        expectations.append((
            "review_text_not_null",
            gx.expectations.ExpectColumnValuesToNotBeNull(column="review_text"),
        ))

    for check_name, expectation in expectations:
        validation = batch.validate(expectation)
        gx_rows.append({
            "table": table_name,
            "check": check_name,
            "success": bool(validation.success),
            "result": str(validation),
        })

gx_report = pd.DataFrame(gx_rows)
gx_report.to_csv(quality_dir / "great_expectations_report.csv", index=False)
display(gx_report[["table", "check", "success"]])
print("Great Expectations report saved to:", (quality_dir / "great_expectations_report.csv").resolve())


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

,table,check,success
0,hotels,table_has_rows,True
1,hotels,hotel_id_not_null,True
2,hotels,hotel_id_unique,True
3,guests,table_has_rows,True
4,guests,guest_id_not_null,True
5,guests,guest_id_unique,True
6,reviews,table_has_rows,True
7,reviews,review_id_not_null,True
8,reviews,review_id_unique,True
9,reviews,review_text_not_null,True


Great Expectations report saved to: C:\Users\telvi\Downloads\AMDARI\montra_guest_exp_sentiment\data\processed\quality_reports\great_expectations_report.csv


In [71]:
# Keep a human-reviewed log of issues, evidence, actions, and any decisions still outstanding.
# Update the evidence/status after reviewing the generated reports.
issue_log = pd.DataFrame([
    {
        "table": "guests",
        "issue": "Missing loyalty_status values",
        "evidence": f"{int(guest['loyalty_status'].isna().sum())} missing values in the extracted guest table",
        "action_taken": "Created guest_clean as a copy and filled missing loyalty_status with 'Unknown'; raw guest table retained",
        "status": "Handled in cleaned copy; validate with business owner",
    },
    {
        "table": "reviews",
        "issue": "Missing or blank review_text",
        "evidence": f"{int(review['review_text'].isna().sum())} null review_text values" if "review_text" in review.columns else "review_text column missing",
        "action_taken": "Reported for review; do not silently delete records",
        "status": "Review missing_values_report.csv and decide treatment",
    },
    {
        "table": "all tables",
        "issue": "Primary-key integrity",
        "evidence": "See primary_key_report.csv and great_expectations_report.csv",
        "action_taken": "Automated null and uniqueness checks added",
        "status": "Review any failed checks before downstream modelling",
    },
    {
        "table": "reviews/bookings/hotels/guests",
        "issue": "Foreign-key integrity",
        "evidence": "See foreign_key_report.csv",
        "action_taken": "Checked child IDs against parent tables",
        "status": "Review orphan IDs, if any, against source system",
    },
])

issue_log.to_csv(quality_dir / "data_quality_issue_log.csv", index=False)
display(issue_log)
print("All quality reports saved under:", quality_dir.resolve())


,table,issue,evidence,action_taken,status
0,guests,Missing loyalty_status values,6036 missing values in the extracted guest table,Created guest_clean as a copy and filled missi...,Handled in cleaned copy; validate with busines...
1,reviews,Missing or blank review_text,0 null review_text values,Reported for review; do not silently delete re...,Review missing_values_report.csv and decide tr...
2,all tables,Primary-key integrity,See primary_key_report.csv and great_expectati...,Automated null and uniqueness checks added,Review any failed checks before downstream mod...
3,reviews/bookings/hotels/guests,Foreign-key integrity,See foreign_key_report.csv,Checked child IDs against parent tables,"Review orphan IDs, if any, against source system"


All quality reports saved under: C:\Users\telvi\Downloads\AMDARI\montra_guest_exp_sentiment\data\processed\quality_reports


## 11. How to interpret and use the results

- **PASS:** the tested rule was satisfied for the current extract.
- **FAIL / non-zero count:** inspect the affected records and confirm whether the rule is valid for this dataset.
- **NOT CHECKED:** a required column was unavailable, so the rule could not be evaluated.
- **Missing value:** may be acceptable for optional fields; document the business decision before filling or dropping it.
- **Duplicate primary key:** usually requires investigation because it can make joins ambiguous.
- **Orphan foreign key:** indicates a child record points to an identifier that was not found in the parent table.
- **Outlier:** is a signal to investigate, not an automatic reason to delete a record.

The CSV reports are evidence for the project's data-quality assessment. Update the issue log only after reviewing the results. Re-run this notebook whenever the source extract or cleaning rules change.